# Prediction

In [1]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import yaml
import torch
from PIL import Image
from segment_anything import sam_model_registry
from tqdm import tqdm
from segment_anything.utils.transforms import ResizeLongestSide

sys.path.append('..')

# Chargement de la configuration MedSAM
config_path = '../configs/config_medsam.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Configuration MedSAM chargée")
print(f"| Device: {device}")

Configuration MedSAM chargée
| Device: cpu


## Données

In [2]:
# Chargement des chemins depuis la config
test_img_path = config['data']['test_img_path']
test_label_path = config['data']['test_label_path']

test_img_files = os.listdir(test_img_path)
test_label_files = os.listdir(test_label_path)

test_img = [os.path.join(test_img_path, file) for file in test_img_files]
test_label = [os.path.join(test_label_path, file) for file in test_label_files]

print(f'Number of testing images: {len(test_img)}')
print(f'Number of testing labels: {len(test_label)}')

Number of testing images: 200
Number of testing labels: 200


In [3]:
def load_img(file_path):
    return Image.open(file_path)

def load_label(file_path):
    # In grayscale (1 canal)
    return Image.open(file_path).convert('L')

## Dataloader test

In [4]:
# !pip install medsam
# !pip install segment-anything
# !pip install timm monai einops

## Dataloader test

## Chargement du modèle MedSAM

In [5]:
# Chargement du modèle MedSAM depuis la config
model_checkpoint = config['model']['model']
model_type = "vit_b"

try:
    medsam_model = sam_model_registry[model_type](checkpoint=None)
    state_dict = torch.load(model_checkpoint, map_location=device)
    medsam_model.load_state_dict(state_dict)
    medsam_model.to(device)
    medsam_model.eval()
    print(f"Modèle MedSAM chargé: {model_checkpoint}")
except Exception as e:
    print(f"Erreur de chargement du modèle: {e}")
    print("Vérifiez que le fichier existe et que segment_anything est bien installé")
    medsam_model = None
    raise RuntimeError(f"Impossible de charger le modèle MedSAM: {e}")

Modèle MedSAM chargé: ../saved_models/medsam/medsam_vit_b.pth


## Fonction de prédiction

In [6]:
def predict_medsam(model, image_path, image_size=config['data']['image_size']):
    if model is None:
        return None
    
    # Charger l'image
    image = Image.open(image_path).convert('RGB')
    original_size = image.size[::-1]  # (H, W)
    image_np = np.array(image)
    
    # Preprocessing
    transform = ResizeLongestSide(image_size)
    image_transformed = transform.apply_image(image_np)
    image_tensor = torch.as_tensor(image_transformed).permute(2, 0, 1).contiguous()
    image_tensor = image_tensor.unsqueeze(0).float().to(device)
    
    # Encoder l'image
    with torch.no_grad():
        image_embedding = model.image_encoder(image_tensor)
    
    # Prompt: boîte englobante complète
    h, w = original_size
    box = torch.tensor([[0, 0, w, h]], dtype=torch.float32).to(device)
    sparse_embeddings, dense_embeddings = model.prompt_encoder(
        points=None,
        boxes=box,
        masks=None,
    )
    
    # Décoder le masque
    with torch.no_grad():
        low_res_masks, _ = model.mask_decoder(
            image_embeddings=image_embedding,
            image_pe=model.prompt_encoder.get_dense_pe(),
            sparse_prompt_embeddings=sparse_embeddings,
            dense_prompt_embeddings=dense_embeddings,
            multimask_output=False,
        )
    
    # Redimensionner à la taille originale
    masks = torch.nn.functional.interpolate(
        low_res_masks,
        size=original_size,
        mode='bilinear',
        align_corners=False
    )
    
    mask = masks.squeeze().cpu().numpy()
    mask = (mask > 0).astype(np.float32)
    
    return mask

In [7]:
import torchvision.transforms as transforms
from src.dataset.dataset import VesselDataset
from torch.utils.data import DataLoader

# Paramètres depuis la config
image_size = config['data']['image_size']

transform = transforms.Compose([
    transforms.Resize((image_size, image_size)),
    transforms.ToTensor(),
])

# Nombre d'images à tester (ajuster selon besoin)
nb_images_to_test = 200

test_dataset = VesselDataset(
    test_img[:nb_images_to_test], 
    test_label[:nb_images_to_test], 
    transform=transform
)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

print(f"Test dataset: {len(test_dataset)} images")

Test dataset: 200 images


## Prédictions

In [9]:
from sklearn.metrics import f1_score
import torch

preds = []
f1_scores = []

print("Génération des prédictions avec MedSAM...")
for i, (images, labels, img_path) in enumerate(tqdm(test_loader)):
    # Obtenir le chemin de l'image originale
    original_img_path = test_img[i]
    
    # Prédire avec MedSAM
    mask = predict_medsam(medsam_model, original_img_path)
    
    if mask is not None:
        # Convertir en tensor
        mask_tensor = torch.from_numpy(mask).unsqueeze(0).unsqueeze(0)
        mask_resized = torch.nn.functional.interpolate(
            mask_tensor, 
            size=labels.shape[-2:],  # Taille du label (H, W)
            mode='bilinear',
            align_corners=False
        )
        preds.append(mask_resized)
        
        # Calculer le F1-score
        label_np = labels.cpu().numpy().flatten()
        mask_np = mask_resized.cpu().numpy().flatten()
        
        # Binariser
        label_binary = (label_np > 0.5).astype(int)
        mask_binary = (mask_np > 0.5).astype(int)
        
        f1 = f1_score(label_binary, mask_binary)
        f1_scores.append(f1)
    
    # Libérer la mémoire
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Convertir la liste en tensor
preds = torch.cat(preds, dim=0)

print(f"Prédictions générées: {len(preds)} images")
print(f"| F1-score moyen: {np.mean(f1_scores):.4f}")
print(f"| F1-score min: {np.min(f1_scores):.4f}")
print(f"| F1-score max: {np.max(f1_scores):.4f}")

Génération des prédictions avec MedSAM...


  8%|▊         | 17/200 [01:51<19:57,  6.55s/it]


KeyboardInterrupt: 

## Visualisation

In [ ]:
# Affichage de plusieurs prédictions
def visualize_predictions(test_imgs, test_labels, predictions, indices=[0, 1, 2, 3]):
    n = len(indices)
    fig, axes = plt.subplots(n, 3, figsize=(15, 5 * n))
    
    for i, idx in enumerate(indices):
        if idx >= len(predictions):
            continue
            
        # Image originale
        axes[i, 0].imshow(load_img(test_imgs[idx]))
        axes[i, 0].set_title(f'Image {idx}')
        axes[i, 0].axis('off')

        # Ground truth
        axes[i, 1].imshow(load_label(test_labels[idx]), cmap='gray')
        axes[i, 1].set_title('Ground truth')
        axes[i, 1].axis('off')

        # Prédiction
        pred = predictions[idx].to('cpu').squeeze().detach().numpy()
        axes[i, 2].imshow(pred, cmap='gray')
        axes[i, 2].set_title('Prédiction')
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()

# 4 premières
visualize_predictions(test_img, test_label, preds, indices=[0, 1, 2, 3])